In [ ]:
from transformers import RobertaTokenizer, TextClassificationPipeline

from toxicity_detection.custom.datasets import current_player_only_point_sep, ChatReplayDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score
from scipy.special import softmax
import pandas as pd

In [ ]:
unlabeled_chat_df = # I cannot provide this data
unlabeled_chat_df['message'] = unlabeled_chat_df['message'].apply(lambda x: str(x))
unlabeled_dset = ChatReplayDataset(chat_df=unlabeled_chat_df, message_concatenator=current_player_only_point_sep, dataset_name=None)

pretrained_model = # I don't think I can provide this model either as it is trained on private data

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('distilbert/distilroberta-base')
tokenizer.padding_side = 'left'
tokenizer.truncation_side = 'left'
pipe = TextClassificationPipeline(model=pretrained_model, tokenizer=tokenizer, top_k=None, function_to_apply=None, device='cuda', padding="max_length", truncation=True)

In [ ]:
def get_predict_probas(clf, dset):
  logits = [[t['score'] for t in sorted(res, key=lambda x: x['label'])] for res in clf(dset)]
  return softmax(logits, axis=1)[:, 1]

def get_metrics(y_true, y_pred):
  return {'acc': float(accuracy_score(y_true, y_pred)), 'bal_acc': balanced_accuracy_score(y_true, y_pred), 'prec': float(precision_score(y_true, y_pred)), 'rec': float(recall_score(y_true, y_pred)), 'bin_f1': float(f1_score(y_true, y_pred))}

In [ ]:
unlabeled_preds =  get_predict_probas(pipe, unlabeled_dset)
unlabeled_chat_df['pred_prob'] = unlabeled_preds
unlabeled_chat_df['pred'] = unlabeled_chat_df['pred_prob'].apply(lambda x: x>.5)
user_tox_map = {u: unlabeled_chat_df[unlabeled_chat_df['user_id']==u]['pred'].mean() for u in set(test_chat_df['user_id'])}

In [ ]:
test_chat_df = # private data

In [ ]:
prop_df = pd.DataFrame.from_dict([{'user_id': u, 'propensity': p, 'messages': len(test_chat_df[test_chat_df['user_id']==u])} for u, p in user_tox_map.items()])
prop_df.sort_values(['propensity', 'messages'], ascending=False)

In [ ]:
prop_df[prop_df['propensity']==0.0].sort_values(['messages'], ascending=False)